# 🏋️ Azure AI Search + Semantic Kernel + AI Agents: Fitness-Fun Workshop 🤸

Welcome to this self-guided workshop where you'll:

1. **Create** an Azure AI Search index containing some sample fitness equipment data
2. **Upload** and verify your documents
3. **Create** a Semantic Kernel Agent (powered by the Foundry Agent Service) that grounds its answers in that index
4. **Run** an asynchronous conversation to query your data (with a fun fitness twist)

> **Note:** This demo uses Semantic Kernel's abstractions over Foundry Agents. For the **New Foundry** (endpoint-based) path you need **semantic-kernel == 1.53.1**:
>
> ```bash
> pip install "semantic-kernel[azure]==1.53.1"
> ```

Also ensure you've set these environment variables:

- `PROJECT_ENDPOINT`
- `MODEL_DEPLOYMENT_NAME`
- `SEARCH_ENDPOINT`, `SEARCH_API_KEY` (for creating, populating, and querying the index)

Let's get started!

## Prerequisites

Before running the cells below, please verify:

1. You have installed the required dependency (New Foundry needs exactly **1.53.1**):
   ```bash
   pip install "semantic-kernel[azure]==1.53.1"
   ```
2. Your environment is configured with `PROJECT_ENDPOINT`, `MODEL_DEPLOYMENT_NAME`, `SEARCH_ENDPOINT`, and `SEARCH_API_KEY`.

## 1. Create & Populate Azure AI Search Index

In this section we will:

1. **Create** an Azure AI Search index called `myfitnessindex` with a schema suited for fitness items
2. **Upload** sample documents containing fitness equipment data
3. **Verify** that the documents are searchable

Make sure your environment has the appropriate search credentials (typically obtained via your AI Foundry project).

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.search.documents import SearchClient

# Load environment variables
notebook_path = Path().absolute()
env_path = notebook_path.parent.parent / '.env'  # Adjust path as needed
load_dotenv(env_path)

# Azure AI Search endpoint + admin key (from the Search resource in the Azure portal)
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_credential = AzureKeyCredential(os.environ["SEARCH_API_KEY"])

# Define the index name for our fitness data
index_name = "myfitnessindex"

try:
    index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)
    print("✅ Created SearchIndexClient")

    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )
    print("✅ Created SearchClient for document operations")
except Exception as e:
    print(f"❌ Error creating search clients: {e}")

### Define the Index Schema

We will create an index with the following fields:

- `FitnessItemID`: Unique key
- `Name`: Searchable text field (also filterable)
- `Category`: Searchable, filterable, and facetable (e.g. Strength, Cardio, Flexibility)
- `Price`: Numeric field (filterable, sortable, and facetable)
- `Description`: Full-text searchable field

In [ ]:
def create_fitness_index():
    fields = [
        SimpleField(name="FitnessItemID", type=SearchFieldDataType.String, key=True),
        SearchableField(name="Name", type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="Category", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SimpleField(name="Price", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
        SearchableField(name="Description", type=SearchFieldDataType.String)
    ]

    index = SearchIndex(name=index_name, fields=fields)

    # Delete the index if it already exists (for a fresh start)
    if index_name in [x.name for x in index_client.list_indexes()]:
        index_client.delete_index(index_name)
        print(f"🗑️ Deleted existing index: {index_name}")

    created = index_client.create_index(index)
    print(f"🎉 Created index: {created.name}")

# Create the index
create_fitness_index()

### Upload Sample Documents

Now we’ll add some sample fitness items to `myfitnessindex`.

In [ ]:
def upload_fitness_docs():
    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )

    sample_docs = [
        {
            "FitnessItemID": "1",
            "Name": "Adjustable Dumbbell",
            "Category": "Strength",
            "Price": 59.99,
            "Description": "A compact, adjustable weight for targeted muscle workouts."
        },
        {
            "FitnessItemID": "2",
            "Name": "Yoga Mat",
            "Category": "Flexibility",
            "Price": 25.0,
            "Description": "Non-slip mat designed for yoga, Pilates, and other exercises."
        },
        {
            "FitnessItemID": "3",
            "Name": "Treadmill",
            "Category": "Cardio",
            "Price": 499.0,
            "Description": "A sturdy treadmill with adjustable speed and incline settings."
        },
        {
            "FitnessItemID": "4",
            "Name": "Resistance Bands",
            "Category": "Strength",
            "Price": 15.0,
            "Description": "Set of colorful bands for light to moderate resistance workouts."
        }
    ]

    result = search_client.upload_documents(documents=sample_docs)
    print(f"🚀 Upload result: {result}")

upload_fitness_docs()
print("✅ Documents uploaded to search index")

### Verify the Documents

Let’s perform a basic search query (e.g. for items in the **Strength** category) to ensure everything is working.

In [ ]:
results = search_client.search(search_text="Strength", filter=None, top=10)

print("🔍 Search results for 'Strength':")
print("-" * 50)
found_items = False
for doc in results:
    found_items = True
    print(f"Name: {doc['Name']}")
    print(f"Category: {doc['Category']}")
    print(f"Price: ${doc['Price']:.2f}")
    print(f"Description: {doc['Description']}")
    print("-" * 50)

if not found_items:
    print("No matching items found.")

## 2. Create a Semantic Kernel Agent that uses Azure AI Search

In this section we use Semantic Kernel's **Azure AI Agent** abstraction (backed by the Foundry Agent Service) to build a fitness shopping assistant. The agent:

- Runs on your Azure OpenAI model (specified by `MODEL_DEPLOYMENT_NAME`)
- Grounds its answers in **Azure AI Search**: for each question we call a small retrieval function (`retrieve_fitness_items`) that queries `myfitnessindex`, then pass the matching items to the agent as context — this is the *retrieval* step of Retrieval-Augmented Generation (RAG)
- Holds an asynchronous, multi-turn conversation, with Semantic Kernel managing the thread for us

The code uses asynchronous Python (`asyncio`) and Semantic Kernel classes from the `semantic_kernel` package. It requires **semantic-kernel == 1.53.1** for the endpoint-based New Foundry path.

In [ ]:
import asyncio
import logging

from azure.identity.aio import DefaultAzureCredential
from semantic_kernel.agents import AzureAIAgent
from semantic_kernel.contents.chat_message_content import ChatMessageContent
from semantic_kernel.contents.utils.author_role import AuthorRole

logging.basicConfig(level=logging.WARNING)

# New Foundry uses the project ENDPOINT (no connection string). Requires semantic-kernel >= 1.53.1.
project_endpoint = os.environ["PROJECT_ENDPOINT"]
model_deployment_name = os.environ["MODEL_DEPLOYMENT_NAME"]

if not model_deployment_name:
    raise ValueError("🚨 MODEL_DEPLOYMENT_NAME not set in .env")
if not project_endpoint:
    raise ValueError("🚨 PROJECT_ENDPOINT not set in .env")


def retrieve_fitness_items(query: str, k: int = 3) -> str:
    """Retrieve the most relevant fitness items from Azure AI Search for a user query.

    This is the *retrieval* half of Retrieval-Augmented Generation (RAG): we query the
    index we populated above and return the matching items as plain text, which we then
    hand to the Semantic Kernel agent as grounding context.
    """
    hits = search_client.search(search_text=query, top=k)
    lines = [
        f"- {d['Name']} ({d['Category']}, ${d['Price']:.2f}): {d['Description']}"
        for d in hits
    ]
    return "\n".join(lines) if lines else "No matching items found."


async with (
    DefaultAzureCredential() as creds,
    # Semantic Kernel builds the Foundry Agents client from the project endpoint (endpoint-based, GA path)
    AzureAIAgent.create_client(credential=creds, endpoint=project_endpoint) as client,
):
    # Create the Foundry agent definition for a fitness shopping assistant
    agent_definition = await client.agents.create_agent(
        model=model_deployment_name,
        name="sk-fitness-shopping-assistant",
        instructions="""
            You are a Fitness Shopping Assistant. Use ONLY the product context provided in each
            message to recommend fitness equipment. Cite the item names you used, and always add a
            short disclaimer that you are not providing medical advice.
        """,
    )

    # Wrap the Foundry agent in a Semantic Kernel Azure AI Agent
    agent = AzureAIAgent(client=client, definition=agent_definition)

    # Semantic Kernel manages the conversation thread for us (created on first invoke)
    thread = None

    user_queries = [
        "Which items are best for strength training?",
        "I need something for cardio under $300. Any suggestions?",
    ]

    try:
        for query in user_queries:
            # RAG step: retrieve relevant items from Azure AI Search, then hand them to the agent
            context = retrieve_fitness_items(query)
            augmented = f"Product context from the fitness catalog:\n{context}\n\nQuestion: {query}"

            print(f"\n# User: {query}\n")
            async for response in agent.invoke(
                messages=ChatMessageContent(role=AuthorRole.USER, content=augmented),
                thread=thread,
            ):
                thread = response.thread
                if response.role != AuthorRole.TOOL:
                    print(f"# Agent: {response.content}\n")
    finally:
        # Clean up the conversation thread and the agent
        if thread is not None:
            await thread.delete()
        await client.agents.delete_agent(agent_definition.id)
        print("🗑️ Cleaned up agent and thread")

### 👀 Watch the agent appear in the Foundry portal

While the cell above runs, the Semantic Kernel agent (`sk-fitness-shopping-assistant`) is created as a real resource in your Foundry project and then cleaned up automatically at the end of the `async` block. To see agents listed in the portal, open the [Microsoft Foundry portal](https://ai.azure.com), select your project, and go to **Build → Agents**. (The agents you created in the earlier Agent Service notebooks are the easiest to catch there, since those notebooks pause before deleting.)

## 3. Cleanup

For this demo we already clean up the agent and thread inside the async function. In case you want to remove the search index as well (for a fresh start), run the cell below.

In [ ]:
try:
    index_client.delete_index(index_name)
    print(f"🗑️ Deleted index {index_name}")
except Exception as e:
    print(f"Error deleting index: {e}")

# 🎉 Congrats!

You've successfully:

1. Created an Azure AI Search index and populated it with fitness data
2. Verified the data via a basic search query
3. Built and run a Semantic Kernel Agent that leverages Azure AI Search to answer natural language queries

Feel free to explore further enhancements (e.g. integrating more tools or advanced evaluation) and enjoy your journey with Azure AI Foundry and Semantic Kernel!